Разработайте модель прогнозирования вероятности согласия клиента на кредитное предложение. Модель должна оценивать, примет ли клиент конкретный оффер по кредитному продукту: бизнес-кредиту, возобновляемой кредитной линии или овердрафту.

Для решения задачи вам предоставляются данные:

*   train_apps.csv – файл с обучающими табличными данными;
*   test_apps.csv – файл с тестовой выборкой.

Файл с тренировочными данными содержит информацию о заявках на кредитные продукты, включая параметры заявки, характеристики клиента и его финансовую активность в банке.

Пример сабмита для отправки на платформу представлен в файле sample_submission.csv.

Инструкция по загрузке решения:

1.   Выберите файл сабмита и загрузите на платформу.
2.   Нажмите кнопку "Отправить на проверку". Если проверка не началась, подождите до 1 минуты (файл может быть тяжелым).

Обработка обычно занимает до 5 минут. Результат появится автоматически, обновлять страницу не нужно.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

Примонтировала свой диск к блокноту, чтобы брать данные напрямую из диска, не храня на своем компьютере

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
train_path = "/content/drive/MyDrive/credit_offer_alpha/train_apps.csv"
test_path = "/content/drive/MyDrive/credit_offer_alpha/test_apps.csv"

Пробуем логистическую регрессию, как мне кажется, для таких задач она подходит

In [ ]:
train_apps = pd.read_csv(train_path)
val_apps = pd.read_csv(test_path)

In [ ]:
X = train_apps.drop(columns=["target_value"])
y = train_apps["target_value"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

Обработаем отдельно даты - колонку decision_day

In [ ]:
X_train["decision_day"] = pd.to_datetime(X_train["decision_day"])
X_test["decision_day"] = pd.to_datetime(X_test["decision_day"])

In [ ]:
for df in [X_train, X_test]:
    df["Year"] = df["decision_day"].dt.year
    df["Month"] = df["decision_day"].dt.month
    df["Day"] = df["decision_day"].dt.day
    df["DayOfWeek"] = df["decision_day"].dt.dayofweek
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

In [ ]:
X_train = X_train.drop(columns=["decision_day"])
X_test = X_test.drop(columns=["decision_day"])

Обработаем отдельно категориальные фичи - колонки db_group_last и fl_adminarea

In [ ]:
db_group_last_vals = {key: [] for key in X_train['db_group_last'].unique().tolist()}
fl_adminarea_vals = {key: [] for key in X_train['fl_adminarea'].unique().tolist()}

In [ ]:
for val in X_train['db_group_last'].tolist():
  for key in db_group_last_vals:
    db_group_last_vals[key].append(1 * (key == val))

In [ ]:
for key in db_group_last_vals:
  X_train[f'db_group_last_vals_{key}'] = db_group_last_vals[key]

In [ ]:
for val in X_train['fl_adminarea'].tolist():
  for key in fl_adminarea_vals:
    fl_adminarea_vals[key].append(1 * (key == val))

In [ ]:
for key in fl_adminarea_vals:
  X_train[f'db_group_last_vals_{key}'] = fl_adminarea_vals[key]

/tmp/ipykernel_4651/4022285823.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[f'db_group_last_vals_{key}'] = fl_adminarea_vals[key]


In [ ]:
db_group_last_vals = {key: [] for key in X_test['db_group_last'].unique().tolist()}
fl_adminarea_vals = {key: [] for key in X_test['fl_adminarea'].unique().tolist()}

In [ ]:
for val in X_test['db_group_last'].tolist():
  for key in db_group_last_vals:
    db_group_last_vals[key].append(1 * (key == val))

In [ ]:
for key in db_group_last_vals:
  X_test[f'db_group_last_vals_{key}'] = db_group_last_vals[key]

In [ ]:
for val in X_test['fl_adminarea'].tolist():
  for key in fl_adminarea_vals:
    fl_adminarea_vals[key].append(1 * (key == val))

In [ ]:
for key in fl_adminarea_vals:
  X_test[f'db_group_last_vals_{key}'] = fl_adminarea_vals[key]

/tmp/ipykernel_4651/3218695551.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_test[f'db_group_last_vals_{key}'] = fl_adminarea_vals[key]


In [ ]:
X_train = X_train.drop(columns=["fl_adminarea", "db_group_last"])
X_test = X_test.drop(columns=["fl_adminarea", "db_group_last"])

Заполним все пропуски медианой

In [ ]:
train_medians = X_train.median()
test_medians = X_test.median()

In [ ]:
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(test_medians)

Масштабирование

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
X_test_scaled = scaler.transform(X_test)
# X_val_scaled = scaler.transform(X_val)

In [ ]:
# Инициализируем и обучаем модель логистической регрессии
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
y_pred_proba

array([0.0217232 , 0.05822128, 0.03718463, ..., 0.07914098, 0.00785869,
       0.03707065])

In [ ]:
# Считаем значение метрики ROC-AUC
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC Score: {roc_auc:.4f}")

ROC-AUC Score: 0.7950


In [ ]:
val_apps.head()

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,...,app_term_mean_360,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea
0,150378,2025-06-05,0.000000,1.480173,1.038845,0.709770,1.602779,1.294844,0.079623,-2.472205,...,0.000000,NaN,NaN,2.653948,-0.629878,-1.667712,0.369900,NaN,inn_scoring,г. Москва
1,194170,2025-06-05,0.961691,2.573324,1.901177,2.306751,1.602779,0.831485,NaN,NaN,...,NaN,NaN,NaN,-1.351879,-0.299958,-6.460294,1.641251,NaN,NaN,г. Москва
2,102106,2025-06-05,-0.623060,-3.719511,-3.062922,1.596982,1.602779,NaN,NaN,NaN,...,0.000000,NaN,NaN,-1.351879,NaN,NaN,NaN,NaN,inn_scoring,NaN
3,256199,2025-06-05,0.000000,1.526705,1.075552,163.069565,1.602779,NaN,NaN,NaN,...,NaN,NaN,NaN,-0.372386,NaN,NaN,NaN,NaN,NaN,Нижегородская область
4,253573,2025-06-05,2.185431,1.998064,1.447383,3.371406,1.602779,NaN,NaN,NaN,...,-2.650641,NaN,NaN,NaN,NaN,NaN,NaN,NaN,inn_scoring,NaN


In [ ]:
train_apps.head()

,front_id,decision_day,loan_amount_last,overdraft_limit_min,overdraft_limit_max,offered_rate,cb_rate,corp_credit_products,sum_deb_ul_90,sum_deb_ul_30,...,overdraft_app_term_max_360,days_from_authperson_registration,fl_hdb_bki_total_active_products,corp_list,count_all_corp_dashboard_events,p75_time_spent_minutes,sum_deb_investment_90,db_group_last,fl_adminarea,target_value
0,127345,2024-02-01,1.339991,-1.847954,-1.586546,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lombard,NaN,0
1,127209,2024-02-01,-2.808489,-3.155914,-2.618329,1.774424,-0.400695,0.842771,NaN,NaN,...,NaN,NaN,NaN,-0.862289,-3.400318,-0.780786,NaN,inn_scoring,NaN,0
2,272776,2024-02-01,2.185431,3.167063,2.369547,-0.709770,-0.400695,0.000000,0.834373,4.897257,...,NaN,NaN,NaN,1.168810,3.015012,0.554064,NaN,NaN,NaN,0
3,127210,2024-02-01,-1.468500,-2.595950,-2.176602,1.774424,-0.400695,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,164500,2024-02-01,0.845440,4.559196,3.467730,-2.484194,-0.400695,0.000000,-0.518122,-3.435251,...,NaN,1.067447,NaN,1.022327,1.506380,0.190096,NaN,NaN,NaN,0


Получим значения на валидационной выборке val_apps

1. обработка дат

In [ ]:
val_apps["decision_day"] = pd.to_datetime(val_apps["decision_day"])

In [ ]:
for df in [val_apps]:
    df["Year"] = df["decision_day"].dt.year
    df["Month"] = df["decision_day"].dt.month
    df["Day"] = df["decision_day"].dt.day
    df["DayOfWeek"] = df["decision_day"].dt.dayofweek
    df["IsWeekend"] = df["DayOfWeek"].isin([5, 6]).astype(int)

In [ ]:
val_apps = val_apps.drop(columns=["decision_day"])

Обработаем категориальные колонки

In [ ]:
db_group_last_vals = {key: [] for key in val_apps['db_group_last'].unique().tolist()}
fl_adminarea_vals = {key: [] for key in val_apps['fl_adminarea'].unique().tolist()}

In [ ]:
for val in val_apps['db_group_last'].tolist():
  for key in db_group_last_vals:
    db_group_last_vals[key].append(1 * (key == val))

In [ ]:
for key in db_group_last_vals:
  val_apps[f'db_group_last_vals_{key}'] = db_group_last_vals[key]

In [ ]:
for val in val_apps['fl_adminarea'].tolist():
  for key in fl_adminarea_vals:
    fl_adminarea_vals[key].append(1 * (key == val))

In [ ]:
for key in fl_adminarea_vals:
  val_apps[f'db_group_last_vals_{key}'] = fl_adminarea_vals[key]

In [ ]:
val_apps = val_apps.drop(columns=["fl_adminarea", "db_group_last"])

Заполним пропуски медианой трейна

In [ ]:
val_medians = val_apps.median()

In [ ]:
val_apps = val_apps.fillna(val_medians)

Теперь масштабирование

In [ ]:
val_apps = val_apps.reindex(columns=X_test.columns, fill_value=0)
val_apps_scaled = scaler.transform(val_apps)

In [ ]:
# val_apps_scaled = val_apps_scaled.reindex(columns=X_test.columns, fill_value=0)
y_pred_proba = model.predict_proba(val_apps_scaled)

In [ ]:
y_pred_proba[:, 1]

array([0.09120561, 0.05850246, 0.03220705, ..., 0.03509188, 0.04123785,
       0.01347345])

In [ ]:
ans = pd.DataFrame({'front_id': val_apps['front_id'], 'target_value': y_pred_proba[:, 1]})

Выгружаем данные

In [ ]:
from google.colab import files

In [ ]:
ans.to_csv("predictions.csv", index=False)

In [ ]:
files.download("predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>